In [43]:
import os
import pandas as pd
from datetime import datetime
import importlib
import numpy as np

np.set_printoptions(suppress=True)

In [44]:
# Relative imports
d = os.path.abspath(os.getcwd())
os.chdir("../..")
import hidden_state_model.processor
importlib.reload(hidden_state_model.processor)
Processor = hidden_state_model.processor.Processor
os.chdir(d)

### Read (and compact) dataframes

In [45]:
compact = True

In [46]:
# Iterate over files in dfs/*.parquet and combine to one df
dfs = []

read = []
data_dir = os.path.join(d, "..", "data")
for file in os.listdir(data_dir):
    fname = os.path.join(data_dir, file)
    if file.endswith(".parquet"):
        read.append(file)
        print(f"Reading {file}")
        df = pd.read_parquet(fname)
        dfs.append(df)
    if file.endswith(".csv"):
        read.append(file)
        df = pd.read_csv(fname, index_col=0)
        dfs.append(df)

raw_df = pd.concat(dfs)

if compact and len(dfs) > 10:
    print("Compacintg dfs")

    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    raw_df.to_parquet(os.path.join(data_dir, f"combined_{timestamp}.parquet"))

    # Move files already in trash to trash within trash
    trash = os.path.join(data_dir, "trash")
    trash_in_trash = os.path.join(trash, f"trash_{timestamp}")
    os.makedirs(trash_in_trash, exist_ok=True)
    for f in os.listdir(trash):
        if f.endswith(".parquet") or f.endswith(".csv"):
            os.rename(os.path.join(trash, f), os.path.join(trash_in_trash, f))
    
    # Move read files to trash and write combined df to dfs/combined_{timestamp}.parquet
    for f in read:
        os.rename(os.path.join(data_dir, f), os.path.join(trash, f))

dfs = []  # Clear memory
raw_df

Reading combined.parquet


,prev_entry,public_cards,player_piles,current_player_i,bet_in_stage,bet_in_game,player_has_played,player_is_folded,first_better_i,big_blind,...,player_type,opponent_names,action,amount,p,relative_ev,rank,tiebreakers,hand_index,state_id
state_id,,,,,,,,,,,,,,,,,,,,,
d2bfcb75-5c63-4875-8244-e93f3e968d9a,None,[],"[98, 96]",0,"[2, 4]","[2, 4]","[False, False]","[False, False]",0,4,...,HumanPlayer,[Max Mekker],call,2,0.36290,0.010887,0,"[1, 0, 0, 0, 0]",75.0,NaN
479e187d-f0d1-495c-b3bb-796e28cccd45,None,[],"[98, 96]",0,"[2, 4]","[2, 4]","[False, False]","[False, False]",0,4,...,HumanPlayer,[Max Mekker],raise,6,0.62230,0.018669,0,"[12, 8, 0, 0, 0]",554.0,NaN
13b12156-078e-4487-bcc6-8755bca2bb35,479e187d-f0d1-495c-b3bb-796e28cccd45,[],"[92, 87]",0,"[8, 13]","[8, 13]","[True, True]","[False, False]",0,4,...,HumanPlayer,[Max Mekker],call,5,0.62230,0.065342,0,"[12, 8, 0, 0, 0]",554.0,NaN
c1ac65df-ef84-49ad-adc5-4042f7c61cff,13b12156-078e-4487-bcc6-8755bca2bb35,"[40, 43, 26]","[87, 87]",0,"[0, 0]","[13, 13]","[False, False]","[False, False]",0,4,...,HumanPlayer,[Max Mekker],check,0,0.43840,0.056992,0,"[12, 8, 4, 1, 0]",554.0,NaN
77be572c-57b6-4b30-95ab-42f176f49a04,c1ac65df-ef84-49ad-adc5-4042f7c61cff,"[40, 43, 26, 23]","[87, 87]",0,"[0, 0]","[13, 13]","[False, False]","[False, False]",0,4,...,HumanPlayer,[Max Mekker],check,0,0.39780,0.051714,0,"[12, 10, 8, 4, 1]",554.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
e45b4fdd-5017-45b8-b839-87b08f5c264d,6573af7b-2d08-4b20-a4c1-2ec5cf5c6c4e,"[27, 28, 1]","[48, 54]",0,"[30, 40]","[44, 54]","[True, True]","[False, False]",0,4,...,HumanPlayer,None,raise,48,0.53380,0.261562,1,"[1, 11, 6, 2, 0]",1153.0,NaN
d1170aa4-3098-4da3-aa8f-ebaef329d236,e45b4fdd-5017-45b8-b839-87b08f5c264d,"[27, 28, 1, 18]","[0, 16]",0,"[0, 0]","[92, 92]","[False, False]","[False, False]",0,4,...,HumanPlayer,None,check,0,0.42460,0.390632,1,"[1, 11, 6, 5, 0]",1153.0,NaN
52c554f1-fb2c-41a6-ad56-de73fe010d10,d1170aa4-3098-4da3-aa8f-ebaef329d236,"[27, 28, 1, 18, 17]","[0, 16]",0,"[0, 0]","[92, 92]","[False, False]","[False, False]",0,4,...,HumanPlayer,None,check,0,0.20560,0.189152,1,"[1, 11, 6, 5, 0]",1153.0,NaN


In [47]:
raw_df.dtypes

prev_entry            object
public_cards          object
player_piles          object
current_player_i       int64
bet_in_stage          object
bet_in_game           object
player_has_played     object
player_is_folded      object
first_better_i         int64
big_blind              int64
player_name           object
player_type           object
opponent_names        object
action                object
amount                 int64
p                    float64
relative_ev          float64
rank                   int64
tiebreakers           object
hand_index           float64
state_id             float64
dtype: object

In [48]:
# Check for conflicting rows
dupe_df = raw_df[raw_df.index.duplicated()]
assert len(dupe_df) == 0, dupe_df

## Process data

In [49]:
processor = Processor(raw_df)
df = processor.get_processed_df()
df

,raise_preflop,raise_flop,raise_turn,raise_river,raise_showdown,call_preflop,call_flop,call_turn,call_river,call_showdown,...,game_id,action,amount,excess_rank,p,relative_ev,stage,player_name,opponent_name,n_players
d2bfcb75-5c63-4875-8244-e93f3e968d9a,0,0,0,0,0,0,0,0,0,0,...,2b5f0cac-2d03-4090-9188-be5b08d7d063,call,2,0,0.36290,0.010887,preflop,Tord,Max Mekker,2
479e187d-f0d1-495c-b3bb-796e28cccd45,0,0,0,0,0,0,0,0,0,0,...,9ed19103-8198-49cd-bd32-afedcfb06b3d,raise,6,0,0.62230,0.018669,preflop,Tord,Max Mekker,2
13b12156-078e-4487-bcc6-8755bca2bb35,6,0,0,0,0,0,0,0,0,0,...,9ed19103-8198-49cd-bd32-afedcfb06b3d,call,5,0,0.62230,0.065342,preflop,Tord,Max Mekker,2
c1ac65df-ef84-49ad-adc5-4042f7c61cff,6,0,0,0,0,5,0,0,0,0,...,9ed19103-8198-49cd-bd32-afedcfb06b3d,check,0,0,0.43840,0.056992,flop,Tord,Max Mekker,2
77be572c-57b6-4b30-95ab-42f176f49a04,6,0,0,0,0,5,0,0,0,0,...,9ed19103-8198-49cd-bd32-afedcfb06b3d,check,0,0,0.39780,0.051714,turn,Tord,Max Mekker,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
e45b4fdd-5017-45b8-b839-87b08f5c264d,10,30,0,0,0,2,0,0,0,0,...,44ce7f72-ee15-440f-8921-312b848145ce,raise,48,0,0.53380,0.261562,flop,Arin,,2
d1170aa4-3098-4da3-aa8f-ebaef329d236,10,78,0,0,0,2,0,0,0,0,...,44ce7f72-ee15-440f-8921-312b848145ce,check,0,0,0.42460,0.390632,turn,Arin,,2
52c554f1-fb2c-41a6-ad56-de73fe010d10,10,78,0,0,0,2,0,0,0,0,...,44ce7f72-ee15-440f-8921-312b848145ce,check,0,0,0.20560,0.189152,river,Arin,,2
8af68bb6-2185-48e6-a703-98c22738aa2f,0,0,0,0,0,0,0,0,0,0,...,21e01334-1ac0-496d-b2ca-8cdaad8580bc,call,2,0,0.46100,0.013830,preflop,Tord,Nina Caliente,2


In [50]:
df.dtypes

raise_preflop                int64
raise_flop                   int64
raise_turn                   int64
raise_river                  int64
raise_showdown               int64
call_preflop                 int64
call_flop                    int64
call_turn                    int64
call_river                   int64
call_showdown                int64
check_preflop                int64
check_flop                   int64
check_turn                   int64
check_river                  int64
check_showdown               int64
opponent_raise_preflop       int64
opponent_raise_flop          int64
opponent_raise_turn          int64
opponent_raise_river         int64
opponent_raise_showdown      int64
opponent_call_preflop        int64
opponent_call_flop           int64
opponent_call_turn           int64
opponent_call_river          int64
opponent_call_showdown       int64
opponent_check_preflop       int64
opponent_check_flop          int64
opponent_check_turn          int64
opponent_check_river

## Training

In [51]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [52]:
X = df.drop(["excess_rank", "game_id", "p", "relative_ev"], axis=1)
y = df["excess_rank"]
groups = df["game_id"]  # Group by 'game_id' to ensure no data leakage

In [53]:
X

,raise_preflop,raise_flop,raise_turn,raise_river,raise_showdown,call_preflop,call_flop,call_turn,call_river,call_showdown,...,opponent_check_flop,opponent_check_turn,opponent_check_river,opponent_check_showdown,action,amount,stage,player_name,opponent_name,n_players
d2bfcb75-5c63-4875-8244-e93f3e968d9a,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,call,2,preflop,Tord,Max Mekker,2
479e187d-f0d1-495c-b3bb-796e28cccd45,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,raise,6,preflop,Tord,Max Mekker,2
13b12156-078e-4487-bcc6-8755bca2bb35,6,0,0,0,0,0,0,0,0,0,...,0,0,0,0,call,5,preflop,Tord,Max Mekker,2
c1ac65df-ef84-49ad-adc5-4042f7c61cff,6,0,0,0,0,5,0,0,0,0,...,1,0,0,0,check,0,flop,Tord,Max Mekker,2
77be572c-57b6-4b30-95ab-42f176f49a04,6,0,0,0,0,5,0,0,0,0,...,2,0,0,0,check,0,turn,Tord,Max Mekker,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
e45b4fdd-5017-45b8-b839-87b08f5c264d,10,30,0,0,0,2,0,0,0,0,...,0,0,0,0,raise,48,flop,Arin,,2
d1170aa4-3098-4da3-aa8f-ebaef329d236,10,78,0,0,0,2,0,0,0,0,...,0,0,0,0,check,0,turn,Arin,,2
52c554f1-fb2c-41a6-ad56-de73fe010d10,10,78,0,0,0,2,0,0,0,0,...,0,1,0,0,check,0,river,Arin,,2
8af68bb6-2185-48e6-a703-98c22738aa2f,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,call,2,preflop,Tord,Nina Caliente,2


In [54]:
y

d2bfcb75-5c63-4875-8244-e93f3e968d9a    0
479e187d-f0d1-495c-b3bb-796e28cccd45    0
13b12156-078e-4487-bcc6-8755bca2bb35    0
c1ac65df-ef84-49ad-adc5-4042f7c61cff    0
77be572c-57b6-4b30-95ab-42f176f49a04    0
                                       ..
e45b4fdd-5017-45b8-b839-87b08f5c264d    0
d1170aa4-3098-4da3-aa8f-ebaef329d236    0
52c554f1-fb2c-41a6-ad56-de73fe010d10    0
8af68bb6-2185-48e6-a703-98c22738aa2f    0
384f4f19-6dd7-4611-ba53-de5e0e6a2ad4    0
Name: excess_rank, Length: 4700, dtype: Int64

In [55]:
# Identify categorical columns (excluding 'game_id')
categorical_cols = ["action", "stage", "player_name", "opponent_name"]

# Preprocessing pipeline: OneHotEncoding for categorical and scaling for numerical
preprocessor = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(drop="first"), categorical_cols)],
    remainder="passthrough",
)

# Create the full pipeline with logistic regression
model = Pipeline(
    [
        ("preprocess", preprocessor),
        (
            "classifier",
            LogisticRegression(
                multi_class="multinomial", solver="lbfgs", max_iter=10_000
            ),
        ),
    ]
)

In [56]:
# Grouped train-test split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

Train shape: (3775, 36)
Test shape: (925, 36)


In [57]:
# Train the model
model.fit(X_train, y_train)

# Evaluate the model
accuracy = model.score(X_test, y_test)
print(f"Accuracy: {accuracy}")

Accuracy: 0.7718918918918919


In [58]:
probabilities = model.predict_proba(X_test)
prob_df = pd.DataFrame(probabilities, columns=model.classes_, index=X_test.index)
prob_df["true"] = y_test.values
prob_df["pred"] = model.predict(X_test)
prob_df["correct"] = prob_df["true"] == prob_df["pred"]
prob_df["goodness"] = prob_df.apply(lambda x: x.get(x["true"]) or 0, axis=1)
print("Accuracy", prob_df["correct"].mean())
print("Mean goodness: ", prob_df["goodness"].mean())
prob_df

Accuracy 0.7718918918918919
Mean goodness:  0.694013005251592


/var/folders/4_/zn5f20ls53v5r1bw3w8dbh0h0000gn/T/ipykernel_82688/1891827510.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  prob_df["goodness"] = prob_df.apply(lambda x: x.get(x["true"]) or 0, axis=1)


,0.0,1.0,2.0,3.0,4.0,5.0,true,pred,correct,goodness
d2bfcb75-5c63-4875-8244-e93f3e968d9a,0.959985,0.038225,0.000111,0.000076,0.001411,0.000192,0,0.0,True,0.959985
880581e5-8389-4186-8f30-827c3ffb81ef,0.694001,0.293056,0.008887,0.001248,0.001544,0.001263,0,0.0,True,0.694001
53585779-3b38-4afa-82aa-f62a1f288adf,0.892830,0.106765,0.000002,0.000131,0.000136,0.000135,0,0.0,True,0.892830
d4726d30-c16d-416a-9d80-78cec5b7f81b,0.169605,0.787855,0.001290,0.015923,0.016491,0.008836,1,1.0,True,0.787855
cbe2d04e-7a99-4f63-8af1-856eb8178309,0.949538,0.049853,0.000159,0.000073,0.000171,0.000206,0,0.0,True,0.949538
...,...,...,...,...,...,...,...,...,...,...
3d0b134e-3aa4-40bd-bade-7c2c7974bf42,0.932414,0.066726,0.000353,0.000149,0.000213,0.000145,0,0.0,True,0.932414
d8246e10-4c9f-4061-8880-353777b4b5e4,0.664543,0.316187,0.008626,0.005302,0.002833,0.002509,0,0.0,True,0.664543
a29812a0-8733-4f15-96ef-752a66da089c,0.660581,0.296832,0.009205,0.022620,0.004278,0.006485,0,0.0,True,0.660581
112a363d-dbfd-402e-af77-c141215391ed,0.634359,0.298684,0.004309,0.032949,0.012590,0.017110,0,0.0,True,0.634359


In [59]:
# Look at incorrect predictions
print(prob_df[prob_df["correct"] == False].shape[0], "incorrect predictions:")
prob_df[prob_df["correct"] == False]

211 incorrect predictions:


,0.0,1.0,2.0,3.0,4.0,5.0,true,pred,correct,goodness
c484e6d7-2ef2-4e30-a893-77a7d227957c,0.526703,0.238620,6.868931e-02,0.156549,0.003460,0.005978,1,0.0,False,2.386203e-01
aa708ce6-e2e1-4b78-a839-ce708321a215,0.819073,0.165155,5.254850e-03,0.008796,0.000979,0.000742,1,0.0,False,1.651552e-01
9631a7d6-0c4c-48c8-bb20-de0e69794de8,0.835544,0.139272,2.396137e-03,0.019087,0.001511,0.002190,1,0.0,False,1.392718e-01
79682b71-1891-47a2-92ec-2f2528fa04a0,0.074832,0.383769,3.636379e-02,0.450530,0.011050,0.043455,1,3.0,False,3.837691e-01
9c5d0789-08ba-400f-9e69-59b707e40c54,0.124148,0.709053,9.318387e-02,0.039738,0.012008,0.021870,4,1.0,False,1.200830e-02
...,...,...,...,...,...,...,...,...,...,...
0cf36d20-5e25-46c0-b1c0-d6cb8109ab48,0.568691,0.346034,3.324374e-02,0.006254,0.039797,0.005980,1,0.0,False,3.460344e-01
8a9579a3-d596-41e3-88c4-463c04b3e465,0.517161,0.434861,1.818717e-02,0.003833,0.021559,0.004399,1,0.0,False,4.348607e-01
7618910c-6ae8-4076-aa80-36ac2f2e1d36,0.492884,0.502388,1.580799e-04,0.002840,0.001342,0.000389,0,1.0,False,4.928840e-01
02ec9c72-2647-4dde-ab9f-0e6eef0d82ba,0.308450,0.676306,1.425104e-21,0.011680,0.003560,0.000003,2,1.0,False,1.425104e-21


### Attempt to infer card probabilities from rank and table

In [60]:
from cpp_poker.cpp_poker import Hand, CardCollection, HandRank, CheatSheet

In [61]:
hand_ranks_per_state = []
table_ranks = []
for i, (state_id, row) in enumerate(prob_df.iterrows()):
    raw_row = raw_df.loc[state_id]
    table_cards = CardCollection(raw_row["public_cards"])
    table_rank = table_cards.rank_hand().get_rank()
    excess_hand_ranks_row = table_cards.rank_all_hands()
    hand_ranks_per_state.append(
        [rank.get_rank() - table_rank for rank in excess_hand_ranks_row]
    )
    table_ranks.append(table_rank)

table_ranks_df = pd.DataFrame(table_ranks, index=prob_df.index, columns=["table_rank"])
hand_ranks_df = pd.DataFrame(hand_ranks_per_state, index=prob_df.index)
hand_ranks_df

,0,1,2,3,4,5,6,7,8,9,...,1316,1317,1318,1319,1320,1321,1322,1323,1324,1325
d2bfcb75-5c63-4875-8244-e93f3e968d9a,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
880581e5-8389-4186-8f30-827c3ffb81ef,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
53585779-3b38-4afa-82aa-f62a1f288adf,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
d4726d30-c16d-416a-9d80-78cec5b7f81b,0,0,0,0,1,0,0,0,0,0,...,0,0,1,0,0,1,0,1,0,1
cbe2d04e-7a99-4f63-8af1-856eb8178309,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3d0b134e-3aa4-40bd-bade-7c2c7974bf42,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
d8246e10-4c9f-4061-8880-353777b4b5e4,1,0,1,0,0,0,0,0,0,0,...,1,0,0,0,1,1,1,0,0,0
a29812a0-8733-4f15-96ef-752a66da089c,1,4,1,0,0,0,0,0,0,0,...,1,0,0,0,1,1,1,0,0,0
112a363d-dbfd-402e-af77-c141215391ed,4,4,0,0,4,4,4,4,0,4,...,1,0,0,0,1,1,1,0,0,0


In [62]:
# Get the values of the column indices from hand_ranks_df
column_indices = hand_ranks_df.values

# Use np.arange to create an array of row indices
row_indices = np.arange(hand_ranks_df.shape[0])[:, None]

# Use the row and column indices to index into the DataFrame values
hand_probs_df = pd.DataFrame(
    prob_df.values[row_indices, column_indices],
    index=hand_ranks_df.index,
    columns=hand_ranks_df.columns,
)

# Normalize the hand probabilities to sum to 1 for each row
hand_probs_df = hand_probs_df.div(hand_probs_df.sum(axis=1), axis=0)
hand_probs_df

,0,1,2,3,4,5,6,7,8,9,...,1316,1317,1318,1319,1320,1321,1322,1323,1324,1325
d2bfcb75-5c63-4875-8244-e93f3e968d9a,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799,...,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799
880581e5-8389-4186-8f30-827c3ffb81ef,0.000781,0.000781,0.000781,0.000781,0.000781,0.000781,0.000781,0.000781,0.000781,0.000781,...,0.000781,0.000781,0.000781,0.000781,0.000781,0.000781,0.000781,0.000781,0.000781,0.000781
53585779-3b38-4afa-82aa-f62a1f288adf,0.000795,0.000795,0.000795,0.000795,0.000795,0.000795,0.000795,0.000795,0.000795,0.000795,...,0.000795,0.000795,0.000795,0.000795,0.000795,0.000795,0.000795,0.000795,0.000795,0.000795
d4726d30-c16d-416a-9d80-78cec5b7f81b,0.000342,0.000342,0.000342,0.000342,0.001591,0.000342,0.000342,0.000342,0.000342,0.000342,...,0.000342,0.000342,0.001591,0.000342,0.000342,0.001591,0.000342,0.001591,0.000342,0.001591
cbe2d04e-7a99-4f63-8af1-856eb8178309,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799,...,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3d0b134e-3aa4-40bd-bade-7c2c7974bf42,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798,...,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798
d8246e10-4c9f-4061-8880-353777b4b5e4,0.000451,0.000947,0.000451,0.000947,0.000947,0.000947,0.000947,0.000947,0.000947,0.000947,...,0.000451,0.000947,0.000947,0.000947,0.000451,0.000451,0.000451,0.000947,0.000947,0.000947
a29812a0-8733-4f15-96ef-752a66da089c,0.00048,0.000007,0.00048,0.001069,0.001069,0.001069,0.001069,0.001069,0.001069,0.001069,...,0.00048,0.001069,0.001069,0.001069,0.00048,0.00048,0.00048,0.001069,0.001069,0.001069
112a363d-dbfd-402e-af77-c141215391ed,0.000022,0.000022,0.001097,0.001097,0.000022,0.000022,0.000022,0.000022,0.001097,0.000022,...,0.000517,0.001097,0.001097,0.001097,0.000517,0.000517,0.000517,0.001097,0.001097,0.001097


In [63]:
# Validate that rows sum to 1
hand_probs_df.sum(axis=1)

d2bfcb75-5c63-4875-8244-e93f3e968d9a    1.0
880581e5-8389-4186-8f30-827c3ffb81ef    1.0
53585779-3b38-4afa-82aa-f62a1f288adf    1.0
d4726d30-c16d-416a-9d80-78cec5b7f81b    1.0
cbe2d04e-7a99-4f63-8af1-856eb8178309    1.0
                                       ... 
3d0b134e-3aa4-40bd-bade-7c2c7974bf42    1.0
d8246e10-4c9f-4061-8880-353777b4b5e4    1.0
a29812a0-8733-4f15-96ef-752a66da089c    1.0
112a363d-dbfd-402e-af77-c141215391ed    1.0
384f4f19-6dd7-4611-ba53-de5e0e6a2ad4    1.0
Length: 925, dtype: object

In [65]:
# Look at the max probabilites for different hands
mask = prob_df["pred"] == 2
max_probs = hand_probs_df[mask].max(axis=1)
min_probs = hand_probs_df[mask].min(axis=1)
max_prob_hands = hand_probs_df[mask].idxmax(axis=1)
min_prob_hands = hand_probs_df[mask].idxmin(axis=1)
mean_probs = hand_probs_df[mask].mean(axis=1)
sample_prob_df = pd.DataFrame(
    {
        "max_prob": max_probs,
        "max_prob_hand": max_prob_hands,
        "min_prob": min_probs,
        "min_prob_hand": min_prob_hands,
        "mean_prob": mean_probs,
    }
)
sample_prob_df["max_prob_hand"] = sample_prob_df["max_prob_hand"].apply(lambda x: Hand(x).str())
sample_prob_df["min_prob_hand"] = sample_prob_df["min_prob_hand"].apply(lambda x: Hand(x).str())
sample_prob_df["table"] = raw_df.loc[sample_prob_df.index, "public_cards"].apply(
    lambda x: CardCollection(x).str()
)
sample_prob_df["predicted_excess_rank"] = prob_df.loc[sample_prob_df.index, "pred"]
sample_prob_df["pred_rank"] = (
    sample_prob_df["predicted_excess_rank"]
    + table_ranks_df.loc[sample_prob_df.index, "table_rank"]
)
sample_prob_df["pred_rank_str"] = sample_prob_df["pred_rank"].apply(
    lambda x: HandRank(int(x), []).get_rank_name()
)
sample_prob_df = sample_prob_df.drop(["predicted_excess_rank"], axis=1)
sample_prob_df

,max_prob,max_prob_hand,min_prob,min_prob_hand,mean_prob,table,pred_rank,pred_rank_str
6118230096,0.001159,♥ 6 ♥ 7,0.000042,♥ 6 ♦ 6,0.000754,♥ 5 ♥ 10 ♦ 9 ♣ 6 ♣ 7,2.0,Two Pairs
6087754896,0.002057,♥ 2 ♥ 5,0.000053,♥ 3 ♥ 4,0.000754,♥ 9 ♦ 5 ♣ 2 ♣ K,2.0,Two Pairs
6085606656,0.00317,♥ 2 ♥ 5,0.00002,♥ 3 ♥ 4,0.000754,♥ 9 ♦ 5 ♣ 2 ♣ K,2.0,Two Pairs
6087820048,0.003349,♥ 2 ♥ 5,0.000022,♥ 3 ♥ 4,0.000754,♥ 9 ♦ 5 ♣ 2 ♣ K,2.0,Two Pairs
6137322224,0.011228,♦ K ♠ K,0.000005,♥ 3 ♥ 4,0.000754,♥ 9 ♥ K ♦ 5 ♣ 2 ♣ K,3.0,Three of a Kind


In [66]:
# Deepdive into a specific row
state_id = sample_prob_df.index[0]
row = sample_prob_df.loc[state_id]
most_likely_hands = hand_probs_df.loc[state_id].sort_values(ascending=False).head(5)
least_likely_hands = hand_probs_df.loc[state_id].sort_values(ascending=True).head(5)
print("Table cards:", row["table"])
print("Predicted excess rank:", prob_df.loc[state_id, "pred"])
print("Actual excess rank", df.loc[state_id, "excess_rank"])
print("Most and least likely hands:")
for hand, prob in most_likely_hands.items():
    print(f"{Hand(hand).str()}: {prob*100:.2f}%")
print("...")
for hand, prob in least_likely_hands.items():
    print(f"{Hand(hand).str()}: {prob*100:.2f}%")

Table cards: ♥ 5 ♥ 10 ♦ 9 ♣ 6 ♣ 7 
Predicted excess rank: 2.0
Actual excess rank 1
Most and least likely hands:
♥ 7 ♣ 9 : 0.12%
♦ 5 ♦ 6 : 0.12%
♥ 7 ♥ 9 : 0.12%
♦ 6 ♦ 7 : 0.12%
♦ 7 ♠ 9 : 0.12%
...
♦ 6 ♠ 6 : 0.00%
♥ 9 ♣ 9 : 0.00%
♥ 7 ♦ 7 : 0.00%
♥ 6 ♠ 6 : 0.00%
♦ 10 ♠ 10 : 0.00%


### Attempt to infer winning probabilities from hidden state probabilities

In [ ]:
hand_winning_probs = []
for i, (state_id, row) in enumerate(hand_probs_df.iterrows()):
    print(f"Processing row {i}", flush=True)
    raw_row = raw_df.loc[state_id]
    table_cards = CardCollection(raw_row["public_cards"])
    processed_row = df.loc[state_id]
    n_players = processed_row["n_players"]
    hand_winning_probs.append(CheatSheet.get_all_winning_probabilities(table_cards, n_players, 1000))

hand_winning_probs_df = pd.DataFrame(hand_winning_probs, index=hand_probs_df.index)
hand_winning_probs_df

Processing row 0
Cache is not loaded, loading cache
Setting up signal handlers
Processing row 1
Processing row 2
Processing row 3
Processing row 4
Processing row 5
Processing row 6


: 